# Clase práctica guiada — Aprendizaje Profundo — UNSAM
## RNN, LSTM y GRU: Pronóstico de Series de Tiempo

---

### Objetivos de aprendizaje

Al terminar este notebook vas a poder:

1. **Entender** por qué las redes feedforward y convolucionales no son ideales para datos secuenciales.
2. **Implementar desde cero** (en PyTorch, sin usar `nn.RNN`) el loop interno de una RNN simple, una LSTM y una GRU — exponiendo cada compuerta explícitamente.
3. **Conectar** la implementación manual con las capas de alto nivel de PyTorch (`nn.RNN`, `nn.LSTM`, `nn.GRU`).
4. **Aplicar** estas arquitecturas al problema de pronóstico de temperatura usando el dataset climático de Jena.
5. **Comparar** modelos mediante MAE y visualizaciones de curvas de entrenamiento.

---

> **Consejo didáctico:** La idea no es correr todo mecánicamente, sino *leer, predecir, ejecutar y observar*.  
> Antes de ejecutar cada celda, intentá anticipar qué va a pasar. ¿La red mejora? ¿El error baja? ¿La curva de validación sigue a la de entrenamiento?

In [3]:
import os
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── Reproducibilidad ──────────────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

# ── Estilo de gráficos (mismo que el resto del curso) ────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)

# ── Dispositivo de cómputo ────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")

# ── Flag para modo rápido (True = menos épocas, útil para clase) ─────────────
FAST_MODE = True
print(f"FAST_MODE: {FAST_MODE}  (cambiar a False para entrenamiento completo)")

ModuleNotFoundError: No module named 'torch'

---
## 1. Series de tiempo: conceptos y datos

### ¿Qué es una serie de tiempo?

Una **serie de tiempo** es cualquier conjunto de mediciones tomadas a intervalos regulares. Ejemplos:

| Dominio | Variable | Frecuencia |
|---------|----------|------------|
| Meteorología | Temperatura, presión atmosférica | Cada 10 min |
| Economía | Precio de una acción | Diaria |
| Biología | Latidos del corazón (ECG) | ~300 Hz |
| Logística | Ventas de un producto | Semanal |

Las tareas más comunes sobre series de tiempo son:

- **Pronóstico (forecasting):** usar el pasado para predecir el futuro.
- **Detección de anomalías:** identificar comportamientos inusuales.
- **Clasificación:** asignar etiquetas a segmentos de la serie.

En este notebook nos concentramos en **pronóstico**: dado un historial de las últimas 120 horas de mediciones climáticas, ¿podemos predecir la temperatura 24 horas después?

---

### El dataset climático de Jena

El dataset fue registrado en la estación meteorológica del Instituto Max Planck de Biogeoquímica en Jena, Alemania. Contiene **14 variables** (temperatura, presión, humedad, velocidad del viento, etc.) medidas **cada 10 minutos** entre 2009 y 2016 (~420.000 registros).

In [ ]:
# ── Descarga del dataset ──────────────────────────────────────────────────────
DATA_URL = "https://s3.amazonaws.com/keras-datasets/jena_climate_2009_2016.csv.zip"
DATA_ZIP  = "jena_climate_2009_2016.csv.zip"
DATA_CSV  = "jena_climate_2009_2016.csv"

if not os.path.exists(DATA_CSV):
    print("Descargando dataset de Jena…")
    urllib.request.urlretrieve(DATA_URL, DATA_ZIP)
    with zipfile.ZipFile(DATA_ZIP, "r") as z:
        z.extractall(".")
    print("Listo.")
else:
    print("Dataset ya descargado.")

# ── Lectura ───────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_CSV)
print(f"Forma del dataframe: {df.shape}")
print("Columnas:", list(df.columns))

In [ ]:
# ── Extraer arrays numéricos ──────────────────────────────────────────────────
# Descartamos la columna "Date Time" y guardamos por separado la temperatura (col 1)
raw_data    = df.iloc[:, 1:].to_numpy(dtype=np.float32)  # (N, 14)
temperature = raw_data[:, 1].copy()                       # columna T (degC)

print(f"raw_data shape : {raw_data.shape}")
print(f"temperature shape: {temperature.shape}")

# ── Visualización 1: temperatura a lo largo de los años ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(temperature, lw=0.4, color="steelblue")
axes[0].set_title("Temperatura (°C) — serie completa (~8 años)")
axes[0].set_xlabel("Timestep (cada 10 min)")
axes[0].set_ylabel("°C")

# Solo los primeros 10 días = 10 * 24 * 6 = 1440 puntos
axes[1].plot(temperature[:1440], lw=1.0, color="tomato")
axes[1].set_title("Temperatura (°C) — primeros 10 días")
axes[1].set_xlabel("Timestep (cada 10 min)")
axes[1].set_ylabel("°C")

plt.tight_layout()
plt.show()

**Observá:** en la serie completa se ve la **periodicidad anual** (las temperaturas suben en verano y bajan en invierno del hemisferio norte). En los primeros 10 días se ve la **periodicidad diaria**: noches frías, días más cálidos.

Estas periodicidades en múltiples escalas temporales son características típicas de muchas series de tiempo reales.

---

### Partición temporal (train / val / test)

Con series de tiempo es **crucial** que la partición respete el orden temporal: el modelo se entrena con el pasado y se evalúa con el futuro. Nunca mezcles datos futuros con el entrenamiento.

| Partición | Fracción | Descripción |
|-----------|----------|-------------|
| Entrenamiento | 50% | Pasado lejano |
| Validación    | 25% | Pasado intermedio |
| Test          | 25% | Datos más recientes |

In [ ]:
N = len(raw_data)
num_train = int(0.50 * N)
num_val   = int(0.25 * N)
num_test  = N - num_train - num_val

print(f"Total:         {N:>8,} timesteps")
print(f"Entrenamiento: {num_train:>8,} timesteps (hasta idx {num_train-1})")
print(f"Validación:    {num_val:>8,} timesteps")
print(f"Test:          {num_test:>8,} timesteps")

---
## 2. Preprocesamiento

### Normalización

Cada variable climática está en una escala distinta (ej. la presión en mbar ≈ 1000, la humedad relativa en % ≈ 50-100, etc.). Normalizamos cada columna con la **media y desvío estándar del conjunto de entrenamiento** — nunca usamos información del futuro (val/test) para normalizar.

$$\tilde{x} = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

### Ventanas deslizantes

Para entrenar un modelo de pronóstico necesitamos pares **(entrada, objetivo)**:

- **Entrada:** secuencia de 120 horas (5 días) de las 14 variables, submuestreada a 1 punto por hora (`sampling_rate=6`).
- **Objetivo:** temperatura 24 horas después del último paso de la secuencia.

```
Índice:  0  1  2  3  4 ... 119  →  objetivo en índice 119 + 144
         └─────────────────────┘  delay = 6 * (120 + 24 - 1) = 858
              X (120 timesteps)                   y
```

La ventana se desplaza de a 1 timestep (con `stride=6` para submuestrear de a hora).

In [ ]:
# ── Parámetros de la ventana ──────────────────────────────────────────────────
SAMPLING_RATE   = 6    # 1 muestra cada hora (dataset: 1 cada 10 min → x6)
SEQUENCE_LENGTH = 120  # 5 días = 120 horas de historia
DELAY           = SAMPLING_RATE * (SEQUENCE_LENGTH + 24 - 1)  # = 858
BATCH_SIZE      = 256

# ── Normalización (usando sólo el train) ─────────────────────────────────────
train_raw = raw_data[:num_train]
mean = train_raw.mean(axis=0)
std  = train_raw.std(axis=0)

data_norm = (raw_data - mean) / std  # normalización completa

print("Parámetros de ventana:")
print(f"  sampling_rate   = {SAMPLING_RATE}")
print(f"  sequence_length = {SEQUENCE_LENGTH}  (→ {SEQUENCE_LENGTH} horas = 5 días de historia)")
print(f"  delay           = {DELAY}  (→ objetivo 24 h después del final de cada ventana)")
print(f"  batch_size      = {BATCH_SIZE}")

In [ ]:
class JenaWindowDataset(Dataset):
    """
    Dataset de ventanas deslizantes sobre la serie de Jena.

    Parámetros
    ----------
    data         : array (N, 14) ya normalizado
    temperature  : array (N,) con la temperatura real (en °C, sin normalizar)
    start, end   : índices de la partición (train/val/test)
    sampling_rate: submuestreo (6 → 1 muestra/hora)
    seq_len      : longitud de la ventana de entrada (en horas)
    delay        : desplazamiento hasta el objetivo (en timesteps originales)
    """
    def __init__(self, data, temperature, start, end,
                 sampling_rate=6, seq_len=120, delay=858):
        self.data        = data
        self.temperature = temperature
        self.sr          = sampling_rate
        self.seq_len     = seq_len
        self.delay       = delay

        # Primer índice posible de inicio de una ventana
        # El total de pasos originales que consume la ventana + el objetivo:
        #   seq_len * sr pasos para la entrada + delay pasos para llegar al target
        # → el último target usable es end - 1
        self.indices = list(range(start, end - delay - seq_len * sr + 1))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start_i = self.indices[idx]
        # Submuestreamos: tomamos seq_len puntos separados por sr pasos
        x_indices = range(start_i, start_i + self.seq_len * self.sr, self.sr)
        X = torch.tensor(self.data[list(x_indices)], dtype=torch.float32)
        # El target es la temperatura real (en °C) delay pasos después del inicio
        y = torch.tensor(
            self.temperature[start_i + self.delay], dtype=torch.float32
        )
        return X, y

sr = SAMPLING_RATE   # alias corto para el constructor

train_ds = JenaWindowDataset(data_norm, temperature, 0, num_train,
                              sr, SEQUENCE_LENGTH, DELAY)
val_ds   = JenaWindowDataset(data_norm, temperature, num_train,
                              num_train + num_val, sr, SEQUENCE_LENGTH, DELAY)
test_ds  = JenaWindowDataset(data_norm, temperature, num_train + num_val,
                              N, sr, SEQUENCE_LENGTH, DELAY)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Verificación: obtener un batch de ejemplo
X_sample, y_sample = next(iter(train_loader))
print(f"Shape de X (un batch): {X_sample.shape}  → (batch, seq_len, features)")
print(f"Shape de y (un batch): {y_sample.shape}  → (batch,) temperaturas en °C")

---
## 3. Baseline de sentido común

Antes de entrenar cualquier red, es fundamental establecer un **baseline**: la cota inferior que cualquier modelo útil debe superar.

Para una serie de tiempo de temperatura, hay una heurística muy razonable:

> **"La temperatura dentro de 24 horas será igual a la temperatura ahora."**

¿Por qué es una buena aproximación? Porque la temperatura tiene cierta inercia. Esta heurística no requiere aprendizaje, pero establece un error mínimo que debemos superar.

Usamos el **Error Absoluto Medio** (MAE) como métrica:

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |ŷ_i - y_i|$$

In [ ]:
def evaluate_naive_baseline(loader):
    """
    Predicción: temperatura actual (último timestep de la secuencia) = temperatura futura.
    El índice de la temperatura en las features es 1 (columna T degC).
    Como la entrada está normalizada, hay que desnormalizar: x * std[1] + mean[1].
    """
    total_err  = 0.0
    n_samples  = 0
    temp_mean  = mean[1]
    temp_std   = std[1]

    for X_batch, y_batch in loader:
        # Último timestep, columna 1 (temperatura), desnormalizada
        pred = X_batch[:, -1, 1] * temp_std + temp_mean  # (batch,)
        total_err += torch.abs(pred - y_batch).sum().item()
        n_samples += y_batch.size(0)

    return total_err / n_samples

baseline_val_mae  = evaluate_naive_baseline(val_loader)
baseline_test_mae = evaluate_naive_baseline(test_loader)

print(f"Baseline de sentido común → Validación MAE: {baseline_val_mae:.2f} °C")
print(f"Baseline de sentido común → Test       MAE: {baseline_test_mae:.2f} °C")
print()
print("Este es el umbral que nuestros modelos deben superar.")

---
## 4. Primer intento: red densa (MLP)

Antes de recurrir a arquitecturas recurrentes, probemos con una red MLP que simplemente aplana la secuencia de entrada.

**¿Por qué podría fallar?**  
Al aplanar, destruimos el orden temporal: la red no sabe qué es "antes" y qué es "después". Todo queda como un vector sin orden. Además, el modelo no puede generalizar a secuencias de diferente longitud.

Vamos a definir también las funciones de entrenamiento y evaluación que reutilizaremos para todos los modelos.

In [ ]:
# ── Funciones de entrenamiento y evaluación (reutilizables para todos los modelos) ──

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_mae  = 0.0
    n = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = model(X).squeeze(-1)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * y.size(0)
        total_mae  += torch.abs(pred.detach() - y).sum().item()
        n += y.size(0)
    return total_loss / n, total_mae / n


def evaluate(model, loader):
    model.eval()
    total_mae = 0.0
    n = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            pred = model(X).squeeze(-1)
            total_mae += torch.abs(pred - y).sum().item()
            n += y.size(0)
    return total_mae / n


def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3):
    """Entrena un modelo y devuelve el historial de MAE por época."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    history = {"train_mae": [], "val_mae": []}

    for epoch in range(1, epochs + 1):
        train_loss, train_mae = train_one_epoch(model, train_loader, optimizer, criterion)
        val_mae = evaluate(model, val_loader)
        history["train_mae"].append(train_mae)
        history["val_mae"].append(val_mae)
        print(f"  Época {epoch:>3}/{epochs}  |  train MAE: {train_mae:.3f}  |  val MAE: {val_mae:.3f}")

    return history


def plot_history(history, title, baseline_mae=None):
    """Grafica curvas de entrenamiento y validación."""
    epochs = range(1, len(history["train_mae"]) + 1)
    plt.figure(figsize=(9, 4))
    plt.plot(epochs, history["train_mae"], "r--", label="Train MAE")
    plt.plot(epochs, history["val_mae"],   "b",   label="Val MAE")
    if baseline_mae is not None:
        plt.axhline(baseline_mae, color="gray", linestyle=":", label=f"Baseline ({baseline_mae:.2f}°C)")
    plt.title(title)
    plt.xlabel("Época")
    plt.ylabel("MAE (°C)")
    plt.legend()
    plt.tight_layout()
    plt.show()

print("Funciones de entrenamiento definidas.")

In [ ]:
class MLPModel(nn.Module):
    """
    Red densa simple: aplana la secuencia y aplica 2 capas lineales.
    Input: (batch, seq_len=120, features=14) → aplanar → (batch, 120*14=1680)
    """
    def __init__(self, seq_len=120, n_features=14, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                            # (batch, 120*14)
            nn.Linear(seq_len * n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x)  # (batch, 1)


# Entrenamiento
EPOCHS = 5 if FAST_MODE else 10
NUM_FEATURES = raw_data.shape[1]

print(f"=== Modelo: MLP (red densa) — {EPOCHS} épocas ===")
mlp_model   = MLPModel(seq_len=SEQUENCE_LENGTH, n_features=NUM_FEATURES).to(DEVICE)
mlp_history = train_model(mlp_model, train_loader, val_loader, epochs=EPOCHS)

mlp_test_mae = evaluate(mlp_model, test_loader)
print(f"\nMLP Test MAE: {mlp_test_mae:.2f} °C")

plot_history(mlp_history, "MLP (red densa) — MAE por época",
             baseline_mae=baseline_val_mae)

---
## 5. Redes Neuronales Recurrentes (RNN)

### ¿Por qué una RNN?

Las redes densas y convolucionales tratan cada entrada de forma **independiente**: no tienen noción de "lo que pasó antes". Para datos donde el **orden importa** (texto, audio, series de tiempo), esto es una limitación crítica.

Una **Red Neuronal Recurrente (RNN)** mantiene un **estado oculto** $h_t$ que se actualiza a cada paso de tiempo, acumulando información del pasado:

$$\boxed{h_t = \tanh\!\left(W_x \, x_t + W_h \, h_{t-1} + b\right)}$$

donde:
- $x_t \in \mathbb{R}^{d}$ — entrada en el paso $t$
- $h_{t-1} \in \mathbb{R}^{u}$ — estado oculto anterior (memoria)
- $W_x \in \mathbb{R}^{u \times d}$, $W_h \in \mathbb{R}^{u \times u}$ — matrices de pesos
- $b \in \mathbb{R}^u$ — bias

En pseudocódigo:

```
h = zeros(hidden_size)          # estado inicial
for x_t in secuencia:
    h = tanh(W_x @ x_t + W_h @ h + b)   # actualiza el estado
salida = W_out @ h              # solo usamos el estado final para el pronóstico
```

---

### 5.1 Implementación manual del loop RNN (NumPy)

Antes de usar PyTorch, veamos el mecanismo en NumPy puro para que quede claro la mecánica. Esta implementación **no aprende**, sólo ilustra el forward pass.

In [ ]:
# ── Forward pass de RNN simple — implementación en NumPy ─────────────────────
# (solo ilustración, no se entrena)

timesteps     = 20
input_features  = 5
hidden_size   = 8

np.random.seed(42)
inputs = np.random.randn(timesteps, input_features)  # secuencia de entrada

# Pesos aleatorios (normalmente se aprenden con backprop)
W_x = np.random.randn(hidden_size, input_features) * 0.1
W_h = np.random.randn(hidden_size, hidden_size)    * 0.1
b   = np.zeros(hidden_size)

h_t = np.zeros(hidden_size)    # estado inicial = ceros
hidden_states = []

for x_t in inputs:                                    # loop sobre timesteps
    h_t = np.tanh(W_x @ x_t + W_h @ h_t + b)        # ecuación de la RNN
    hidden_states.append(h_t.copy())

hidden_states = np.array(hidden_states)  # (timesteps, hidden_size)

# Visualización: evolución del estado oculto
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(inputs)
axes[0].set_title("Secuencia de entrada (5 features, 20 pasos)")
axes[0].set_xlabel("Paso de tiempo")

im = axes[1].imshow(hidden_states.T, aspect="auto", cmap="viridis")
axes[1].set_title("Estados ocultos (8 neuronas × 20 pasos)")
axes[1].set_xlabel("Paso de tiempo")
axes[1].set_ylabel("Neurona oculta")
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

print(f"Estado oculto final h_T: shape = {h_t.shape}")
print("→ Este vector de 8 valores resume toda la secuencia de 20 pasos.")

### 5.2 RNN manual en PyTorch (sin `nn.RNN`)

Ahora implementamos la misma lógica en PyTorch para que sea **entrenable**. Exponemos el loop explícitamente — exactamente lo que `nn.RNN` hace internamente.

In [ ]:
class ManualRNNModel(nn.Module):
    """
    RNN implementada manualmente con un loop explícito.
    Equivalente a nn.RNN(...) + nn.Linear(...) pero con el loop visible.

    Parámetros:
        input_size  : número de features de entrada (14)
        hidden_size : dimensión del estado oculto
        output_size : dimensión de salida (1 para regresión)
    """
    def __init__(self, input_size, hidden_size, output_size=1):
        super().__init__()
        self.hidden_size = hidden_size

        # W_x: input → hidden
        self.W_x = nn.Linear(input_size, hidden_size, bias=False)
        # W_h: hidden → hidden (recurrent connection)
        self.W_h = nn.Linear(hidden_size, hidden_size, bias=True)
        # Capa de salida
        self.fc  = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        """
        x: (batch, seq_len, input_size)
        Devuelve: (batch, output_size)
        """
        batch_size, seq_len, _ = x.shape

        # Estado oculto inicial = ceros
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)

        # ── Loop recurrente (este es el corazón de la RNN) ────────────────
        for t in range(seq_len):
            x_t = x[:, t, :]                               # (batch, input_size)
            h = torch.tanh(self.W_x(x_t) + self.W_h(h))   # ecuación de la RNN
        # ─────────────────────────────────────────────────────────────────

        # Usamos el estado final h_T para hacer el pronóstico
        return self.fc(h)  # (batch, output_size)


# ── Entrenamiento ─────────────────────────────────────────────────────────────
EPOCHS_RNN = 5 if FAST_MODE else 15
HIDDEN     = 16

print(f"=== Modelo: RNN Manual (hidden={HIDDEN}) — {EPOCHS_RNN} épocas ===")
rnn_manual_model   = ManualRNNModel(NUM_FEATURES, HIDDEN).to(DEVICE)
rnn_manual_history = train_model(rnn_manual_model, train_loader, val_loader,
                                 epochs=EPOCHS_RNN)

rnn_manual_test_mae = evaluate(rnn_manual_model, test_loader)
print(f"\nRNN Manual — Test MAE: {rnn_manual_test_mae:.2f} °C")

plot_history(rnn_manual_history, f"RNN Manual (hidden={HIDDEN}) — MAE por época",
             baseline_mae=baseline_val_mae)

### 5.3 `nn.RNN` de PyTorch

La clase `nn.RNN` hace exactamente lo mismo que implementamos arriba, pero de forma optimizada. Verificamos que produce resultados similares.

In [ ]:
class SimpleRNNModel(nn.Module):
    """
    Usa nn.RNN de PyTorch directamente.
    batch_first=True → espera input (batch, seq_len, features).
    """
    def __init__(self, input_size, hidden_size, output_size=1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc  = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # output: (batch, seq_len, hidden_size)   ← salidas en cada paso
        # h_n:   (1, batch, hidden_size)          ← estado oculto final
        _, h_n = self.rnn(x)
        return self.fc(h_n.squeeze(0))  # (batch, output_size)


EPOCHS_RNN_PT = 5 if FAST_MODE else 15

print(f"=== Modelo: nn.RNN (hidden={HIDDEN}) — {EPOCHS_RNN_PT} épocas ===")
simple_rnn_model   = SimpleRNNModel(NUM_FEATURES, HIDDEN).to(DEVICE)
simple_rnn_history = train_model(simple_rnn_model, train_loader, val_loader,
                                  epochs=EPOCHS_RNN_PT)

simple_rnn_test_mae = evaluate(simple_rnn_model, test_loader)
print(f"\nnn.RNN — Test MAE: {simple_rnn_test_mae:.2f} °C")

plot_history(simple_rnn_history, f"nn.RNN (hidden={HIDDEN}) — MAE por época",
             baseline_mae=baseline_val_mae)

### El problema de los gradientes que desaparecen

¿Por qué la RNN simple tiene dificultades con secuencias largas? Durante el entrenamiento, el gradiente se propaga **hacia atrás en el tiempo** (Backpropagation Through Time, BPTT). En cada paso hacia atrás, el gradiente se **multiplica** por $W_h$ y por la derivada de tanh.

Si los valores propios de $W_h$ son menores que 1, el gradiente se **encoge exponencialmente** → los pasos lejanos no contribuyen al aprendizaje.

$$\frac{\partial \mathcal{L}}{\partial h_0} = \frac{\partial \mathcal{L}}{\partial h_T} \cdot \prod_{t=1}^{T} \frac{\partial h_t}{\partial h_{t-1}} \approx \frac{\partial \mathcal{L}}{\partial h_T} \cdot (W_h)^T$$

Para $T = 120$ pasos, $(W_h)^{120}$ puede acercarse a cero muy rápidamente.

> **¿Qué pasa si intentás?** Aumentá `HIDDEN` a 64 y entrenás más épocas con `FAST_MODE = False`. ¿Mejora significativamente?

La solución a este problema son las arquitecturas con **compuertas**: LSTM y GRU.

---
## 6. LSTM — Long Short-Term Memory

### ¿Qué es una LSTM?

Propuesta por Hochreiter & Schmidhuber (1997), la LSTM soluciona el problema del gradiente que desaparece agregando una segunda "carretera" de información llamada **estado de la celda** $c_t$.

Imaginá una cinta transportadora que corre en paralelo a la secuencia. La información puede "subirse" a la cinta, "bajarse" o continuar sin alterarse. Las **compuertas** (gates) controlan el flujo:

| Compuerta | Símbolo | Función |
|-----------|---------|---------|
| Olvido    | $f_t$   | ¿Cuánto del pasado ($c_{t-1}$) conservar? |
| Entrada   | $i_t$   | ¿Cuánta información nueva agregar? |
| Candidato | $\tilde{c}_t$ | Información nueva propuesta |
| Salida    | $o_t$   | ¿Qué del estado de la celda exponer como $h_t$? |

Las ecuaciones completas (todas usan sigmoide excepto el candidato que usa tanh):

$$f_t = \sigma(W_{xf}\,x_t + W_{hf}\,h_{t-1} + b_f) \quad \text{(forget gate)}$$
$$i_t = \sigma(W_{xi}\,x_t + W_{hi}\,h_{t-1} + b_i) \quad \text{(input gate)}$$
$$\tilde{c}_t = \tanh(W_{xg}\,x_t + W_{hg}\,h_{t-1} + b_g) \quad \text{(candidate)}$$
$$o_t = \sigma(W_{xo}\,x_t + W_{ho}\,h_{t-1} + b_o) \quad \text{(output gate)}$$
$$\boxed{c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t}$$
$$\boxed{h_t = o_t \odot \tanh(c_t)}$$

La clave es la ecuación de $c_t$: si $f_t \approx 1$ e $i_t \approx 0$, la celda **recuerda** sin modificación — el gradiente puede fluir sin atenuarse.

---

### 6.1 Implementación manual de la LSTM (sin `nn.LSTM`)

In [ ]:
class ManualLSTMModel(nn.Module):
    """
    LSTM implementada manualmente: todas las compuertas son explícitas.
    Se usa nn.Linear para cada proyección pero el loop y la lógica de las
    compuertas son visibles.
    """
    def __init__(self, input_size, hidden_size, output_size=1):
        super().__init__()
        self.hidden_size = hidden_size

        # Proyecciones de la entrada para cada compuerta (bias incluido en una sola)
        # Agrupamos en una sola proyección de tamaño 4*hidden para eficiencia
        # pero la separamos luego para mayor claridad
        self.W_x = nn.Linear(input_size,   4 * hidden_size, bias=False)
        self.W_h = nn.Linear(hidden_size,  4 * hidden_size, bias=True)

        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        """
        x: (batch, seq_len, input_size)
        """
        batch_size, seq_len, _ = x.shape
        H = self.hidden_size

        # Estados iniciales: estado oculto h y estado de la celda c
        h = torch.zeros(batch_size, H, device=x.device)
        c = torch.zeros(batch_size, H, device=x.device)

        for t in range(seq_len):
            x_t = x[:, t, :]  # (batch, input_size)

            # Proyección conjunta: (batch, 4*H)
            gates = self.W_x(x_t) + self.W_h(h)

            # Separar las 4 compuertas
            g_f, g_i, g_g, g_o = gates.chunk(4, dim=-1)

            f = torch.sigmoid(g_f)   # forget gate  — ¿cuánto del pasado conservar?
            i = torch.sigmoid(g_i)   # input gate   — ¿cuánta info nueva entra?
            g = torch.tanh(g_g)      # candidate    — propuesta de nueva información
            o = torch.sigmoid(g_o)   # output gate  — ¿qué se expone como h_t?

            # Actualizar estado de la celda (la "cinta transportadora")
            c = f * c + i * g        # c_t = f ⊙ c_{t-1} + i ⊙ g̃
            # Actualizar estado oculto
            h = o * torch.tanh(c)    # h_t = o ⊙ tanh(c_t)

        return self.fc(h)  # (batch, output_size)


# ── Entrenamiento ─────────────────────────────────────────────────────────────
EPOCHS_LSTM = 8 if FAST_MODE else 20

print(f"=== Modelo: LSTM Manual (hidden={HIDDEN}) — {EPOCHS_LSTM} épocas ===")
lstm_manual_model   = ManualLSTMModel(NUM_FEATURES, HIDDEN).to(DEVICE)
lstm_manual_history = train_model(lstm_manual_model, train_loader, val_loader,
                                   epochs=EPOCHS_LSTM)

lstm_manual_test_mae = evaluate(lstm_manual_model, test_loader)
print(f"\nLSTM Manual — Test MAE: {lstm_manual_test_mae:.2f} °C")

plot_history(lstm_manual_history, f"LSTM Manual (hidden={HIDDEN}) — MAE por época",
             baseline_mae=baseline_val_mae)

### 6.2 Visualización de las compuertas LSTM

Un aspecto pedagógico clave: ¿qué aprenden las compuertas? Visualizamos los valores de $f_t$, $i_t$, $o_t$ para una secuencia de ejemplo luego del entrenamiento.

In [ ]:
def get_lstm_gates(model, x_seq):
    """
    Ejecuta un forward pass del ManualLSTMModel guardando los valores de cada
    compuerta en cada paso de tiempo para UNA única secuencia.
    x_seq: (1, seq_len, input_size)
    """
    model.eval()
    H = model.hidden_size
    seq_len = x_seq.shape[1]
    x_seq = x_seq.to(DEVICE)

    h = torch.zeros(1, H, device=DEVICE)
    c = torch.zeros(1, H, device=DEVICE)

    f_list, i_list, o_list, c_list, h_list = [], [], [], [], []

    with torch.no_grad():
        for t in range(seq_len):
            x_t   = x_seq[:, t, :]
            gates = model.W_x(x_t) + model.W_h(h)
            g_f, g_i, g_g, g_o = gates.chunk(4, dim=-1)
            f = torch.sigmoid(g_f)
            i = torch.sigmoid(g_i)
            g = torch.tanh(g_g)
            o = torch.sigmoid(g_o)
            c = f * c + i * g
            h = o * torch.tanh(c)
            f_list.append(f.cpu().numpy())
            i_list.append(i.cpu().numpy())
            o_list.append(o.cpu().numpy())
            c_list.append(c.cpu().numpy())
            h_list.append(h.cpu().numpy())

    return {
        "forget": np.concatenate(f_list, axis=0),   # (seq_len, H)
        "input":  np.concatenate(i_list, axis=0),
        "output": np.concatenate(o_list, axis=0),
        "cell":   np.concatenate(c_list, axis=0),
        "hidden": np.concatenate(h_list, axis=0),
    }


# Tomamos una secuencia de validación
val_iter  = iter(val_loader)
X_val, y_val = next(val_iter)
gates_data = get_lstm_gates(lstm_manual_model, X_val[:1])  # primera secuencia del batch

# Graficamos las 3 compuertas y el estado oculto
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
labels = ["forget", "input", "output", "hidden"]
cmaps  = ["Reds", "Greens", "Blues", "viridis"]

for ax, key, cmap in zip(axes.flat, labels, cmaps):
    im = ax.imshow(gates_data[key].T, aspect="auto", cmap=cmap, vmin=0, vmax=1)
    ax.set_title(f"Compuerta {key} ($f_t$/$i_t$/$o_t$)" if key != "hidden"
                  else "Estado oculto $h_t$")
    ax.set_ylabel("Neurona")
    ax.set_xlabel("Paso de tiempo (horas)")
    plt.colorbar(im, ax=ax)

plt.suptitle("Valores internos de la LSTM a lo largo de la secuencia", y=1.01)
plt.tight_layout()
plt.show()

**Observá las compuertas:**

- **Forget gate ($f_t$):** Si está cercana a 1 → la célula **recuerda** el estado anterior. Si está cerca a 0 → lo **olvida**.
- **Input gate ($i_t$):** Valores altos → la red está **asimilando nueva información** en ese paso.
- **Output gate ($o_t$):** Controla cuánto del estado de la célula se "expone" como $h_t$.

---

### 6.3 `nn.LSTM` de PyTorch con dropout

In [ ]:
class LSTMModel(nn.Module):
    """
    LSTM usando nn.LSTM de PyTorch.
    Incorpora dropout sobre la salida de la capa recurrente.
    """
    def __init__(self, input_size, hidden_size, output_size=1, dropout=0.2):
        super().__init__()
        self.lstm    = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # h_n: (num_layers=1, batch, hidden_size) — estado oculto final
        _, (h_n, _) = self.lstm(x)
        h = self.dropout(h_n.squeeze(0))  # (batch, hidden_size)
        return self.fc(h)


EPOCHS_LSTM_PT = 10 if FAST_MODE else 25

print(f"=== Modelo: nn.LSTM (hidden={HIDDEN}, dropout=0.2) — {EPOCHS_LSTM_PT} épocas ===")
lstm_model   = LSTMModel(NUM_FEATURES, HIDDEN, dropout=0.2).to(DEVICE)
lstm_history = train_model(lstm_model, train_loader, val_loader,
                            epochs=EPOCHS_LSTM_PT)

lstm_test_mae = evaluate(lstm_model, test_loader)
print(f"\nnn.LSTM — Test MAE: {lstm_test_mae:.2f} °C")

plot_history(lstm_history, f"nn.LSTM (hidden={HIDDEN}, dropout=0.2) — MAE por época",
             baseline_mae=baseline_val_mae)

---
## 7. GRU — Gated Recurrent Unit

### ¿Qué es una GRU?

Propuesta por Cho et al. (2014), la GRU es una **simplificación de la LSTM** que fusiona las compuertas de olvido y entrada en una sola llamada **compuerta de actualización** ($z_t$), y elimina el estado de celda separado $c_t$.

Resultado: **menos parámetros** que la LSTM con rendimiento similar en muchos problemas.

#### Ecuaciones de la GRU:

$$z_t = \sigma(W_{xz}\,x_t + W_{hz}\,h_{t-1} + b_z) \quad \text{(update gate — ¿cuánto actualizar?)}$$
$$r_t = \sigma(W_{xr}\,x_t + W_{hr}\,h_{t-1} + b_r) \quad \text{(reset gate — ¿cuánto del pasado usar?)}$$
$$\tilde{h}_t = \tanh\!\left(W_{x\tilde{h}}\,x_t + W_{h\tilde{h}}\,(r_t \odot h_{t-1}) + b_{\tilde{h}}\right) \quad \text{(candidate)}$$
$$\boxed{h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t}$$

La ecuación de $h_t$ es elegante: si $z_t \approx 0$ → la GRU **recuerda** ($h_t \approx h_{t-1}$); si $z_t \approx 1$ → **actualiza** con nueva información.

#### Comparación LSTM vs GRU

| | LSTM | GRU |
|-|------|-----|
| Estados | $h_t$, $c_t$ | $h_t$ |
| Compuertas | 3 (i, f, o) | 2 (z, r) |
| Parámetros | $4 \times (d+u) \times u$ | $3 \times (d+u) \times u$ |
| Rendimiento | Mejor en secuencias muy largas | Similar, más rápida |

---

### 7.1 Implementación manual de la GRU (sin `nn.GRU`)

In [ ]:
class ManualGRUModel(nn.Module):
    """
    GRU implementada manualmente con todas las compuertas explícitas.
    """
    def __init__(self, input_size, hidden_size, output_size=1):
        super().__init__()
        self.hidden_size = hidden_size

        # Proyecciones de entrada para las 3 operaciones (z, r, h̃)
        self.W_xz = nn.Linear(input_size,  hidden_size, bias=False)
        self.W_xr = nn.Linear(input_size,  hidden_size, bias=False)
        self.W_xh = nn.Linear(input_size,  hidden_size, bias=False)

        # Proyecciones recurrentes
        self.W_hz = nn.Linear(hidden_size, hidden_size, bias=True)
        self.W_hr = nn.Linear(hidden_size, hidden_size, bias=True)
        self.W_hh = nn.Linear(hidden_size, hidden_size, bias=True)

        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        """
        x: (batch, seq_len, input_size)
        """
        batch_size, seq_len, _ = x.shape
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)

        for t in range(seq_len):
            x_t = x[:, t, :]

            # Update gate: ¿cuánto actualizar el estado?
            z = torch.sigmoid(self.W_xz(x_t) + self.W_hz(h))

            # Reset gate: ¿cuánto del pasado usar para calcular h̃?
            r = torch.sigmoid(self.W_xr(x_t) + self.W_hr(h))

            # Candidato: nueva información propuesta
            h_tilde = torch.tanh(self.W_xh(x_t) + self.W_hh(r * h))

            # Actualización del estado oculto (mezcla pasado y nuevo)
            h = (1 - z) * h + z * h_tilde

        return self.fc(h)


# ── Entrenamiento ─────────────────────────────────────────────────────────────
EPOCHS_GRU = 8 if FAST_MODE else 20

print(f"=== Modelo: GRU Manual (hidden={HIDDEN}) — {EPOCHS_GRU} épocas ===")
gru_manual_model   = ManualGRUModel(NUM_FEATURES, HIDDEN).to(DEVICE)
gru_manual_history = train_model(gru_manual_model, train_loader, val_loader,
                                  epochs=EPOCHS_GRU)

gru_manual_test_mae = evaluate(gru_manual_model, test_loader)
print(f"\nGRU Manual — Test MAE: {gru_manual_test_mae:.2f} °C")

plot_history(gru_manual_history, f"GRU Manual (hidden={HIDDEN}) — MAE por época",
             baseline_mae=baseline_val_mae)

### 7.2 GRU apilada con `nn.GRU`

Apilar múltiples capas recurrentes aumenta la capacidad del modelo. La primera capa aprende patrones de bajo nivel (hora a hora), la segunda aprende relaciones entre esos patrones.

Para apilar capas, la capa intermedia debe devolver **todas** las salidas (no solo la final) usando `return_sequences=True` — en PyTorch eso equivale a usar la salida completa `output` (no solo `h_n`).

In [ ]:
class StackedGRUModel(nn.Module):
    """
    GRU apilada (2 capas) con dropout.
    La primera capa devuelve la secuencia completa (num_layers=2 en nn.GRU
    ya hace esto internamente, pero lo modelamos con 2 instancias separadas
    para que el diagrama de datos sea transparente).
    """
    def __init__(self, input_size, hidden_size, output_size=1, dropout=0.3):
        super().__init__()
        # Primera capa GRU: recibe la secuencia, devuelve todas las salidas
        self.gru1    = nn.GRU(input_size,   hidden_size, batch_first=True)
        self.drop1   = nn.Dropout(dropout)
        # Segunda capa GRU: recibe las salidas de la primera
        self.gru2    = nn.GRU(hidden_size,  hidden_size, batch_first=True)
        self.drop2   = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Capa 1: output=(batch, seq_len, hidden), h_n=(1, batch, hidden)
        out1, _ = self.gru1(x)
        out1    = self.drop1(out1)              # dropout a toda la secuencia
        # Capa 2: toma la secuencia completa de la capa 1
        out2, h_n = self.gru2(out1)
        h_last  = self.drop2(h_n.squeeze(0))   # estado final: (batch, hidden)
        return self.fc(h_last)


EPOCHS_GRU_PT = 10 if FAST_MODE else 25

print(f"=== Modelo: GRU Apilada (2×hidden={HIDDEN}, dropout=0.3) — {EPOCHS_GRU_PT} épocas ===")
stacked_gru_model   = StackedGRUModel(NUM_FEATURES, HIDDEN, dropout=0.3).to(DEVICE)
stacked_gru_history = train_model(stacked_gru_model, train_loader, val_loader,
                                   epochs=EPOCHS_GRU_PT)

stacked_gru_test_mae = evaluate(stacked_gru_model, test_loader)
print(f"\nGRU Apilada — Test MAE: {stacked_gru_test_mae:.2f} °C")

plot_history(stacked_gru_history,
             f"GRU Apilada 2 capas (hidden={HIDDEN}, dropout=0.3) — MAE por época",
             baseline_mae=baseline_val_mae)

---
## 8. Comparación final de modelos

Reunimos todos los resultados en una tabla y en un gráfico comparativo.

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

results = [
    ("Baseline sentido común",  None,               baseline_val_mae,      baseline_test_mae,    0),
    ("MLP (densa)",             mlp_history,        min(mlp_history["val_mae"]),        mlp_test_mae,         count_params(mlp_model)),
    ("RNN Manual",              rnn_manual_history, min(rnn_manual_history["val_mae"]), rnn_manual_test_mae,  count_params(rnn_manual_model)),
    ("nn.RNN",                  simple_rnn_history, min(simple_rnn_history["val_mae"]), simple_rnn_test_mae,  count_params(simple_rnn_model)),
    ("LSTM Manual",             lstm_manual_history,min(lstm_manual_history["val_mae"]),lstm_manual_test_mae, count_params(lstm_manual_model)),
    ("nn.LSTM + dropout",       lstm_history,       min(lstm_history["val_mae"]),       lstm_test_mae,        count_params(lstm_model)),
    ("GRU Manual",              gru_manual_history, min(gru_manual_history["val_mae"]), gru_manual_test_mae,  count_params(gru_manual_model)),
    ("GRU Apilada + dropout",   stacked_gru_history,min(stacked_gru_history["val_mae"]),stacked_gru_test_mae, count_params(stacked_gru_model)),
]

print(f"{'Modelo':<28} {'Mejor Val MAE':>14} {'Test MAE':>10} {'Parámetros':>12}")
print("-" * 68)
for name, _, val_mae, test_mae, params in results:
    params_str = f"{params:,}" if params > 0 else "—"
    print(f"{name:<28} {val_mae:>14.3f} {test_mae:>10.3f} {params_str:>12}")

In [ ]:
# ── Gráfico comparativo de val MAE por época ─────────────────────────────────
colors = ["C0", "C1", "C2", "C3", "C4", "C5"]
plt.figure(figsize=(12, 5))

for (name, hist, _, _, _), color in zip(results[1:], colors):  # saltamos baseline
    if hist is not None:
        plt.plot(range(1, len(hist["val_mae"]) + 1), hist["val_mae"],
                 label=name, color=color)

plt.axhline(baseline_val_mae, color="gray", linestyle=":", lw=2,
            label=f"Baseline ({baseline_val_mae:.2f} °C)")
plt.xlabel("Época")
plt.ylabel("Validación MAE (°C)")
plt.title("Comparación de modelos — Validación MAE por época")
plt.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Pronósticos reales vs predichos (modelo LSTM) ────────────────────────────
lstm_model.eval()
preds, trues = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(DEVICE)
        p = lstm_model(X_b).squeeze(-1).cpu().numpy()
        preds.extend(p)
        trues.extend(y_b.numpy())
    if len(preds) > 500:
        preds = preds[:500]
        trues = trues[:500]

plt.figure(figsize=(12, 4))
plt.plot(trues, label="Real (°C)", lw=1.2, alpha=0.8)
plt.plot(preds, label="LSTM predicción (°C)", lw=1.2, alpha=0.8)
plt.title("Pronóstico de temperatura — LSTM vs Real (primeros 500 ejemplos del test)")
plt.xlabel("Ejemplo de test")
plt.ylabel("Temperatura (°C)")
plt.legend()
plt.tight_layout()
plt.show()

---
## 9. Conclusiones

### ¿Qué aprendimos?

#### Sobre las arquitecturas

| Arquitectura | Fortaleza | Limitación |
|---|---|---|
| **MLP** | Simple, rápida | Destruye el orden temporal al aplanar |
| **RNN** | Modela secuencias | Gradientes que desaparecen en secuencias largas |
| **LSTM** | Maneja dependencias largas con la celda $c_t$ | Más parámetros (~4× más que RNN) |
| **GRU** | Balance entre capacidad y eficiencia | Ligeramente menos expresiva que LSTM en algunos casos |

#### Lecciones clave

1. **Establecer un baseline primero.** La heurística "la temperatura no cambia" es difícil de superar — esto indica que el problema tiene una componente de inercia fuerte.

2. **Las RNNs preservan el orden.** A diferencia del MLP, la RNN, LSTM y GRU mantienen la información temporal explícitamente.

3. **Las compuertas son la solución al gradiente que desaparece.** Las multiplicaciones de $f_t \odot c_{t-1}$ (LSTM) o $(1-z_t) \odot h_{t-1}$ (GRU) crean "autopistas" por donde el gradiente puede fluir sin atenuarse.

4. **Implementar desde cero vale la pena.** Al ver el loop explícito y cada compuerta, entendemos qué hace realmente `nn.LSTM` — y podemos modificarlo cuando sea necesario.

---

### ¿Cómo seguir?

- **Bidirectional RNNs:** procesar la secuencia en ambas direcciones (útil para NLP, no tanto para pronóstico).
- **Attention y Transformers:** la arquitectura que reemplazó a las RNNs en la mayoría de los problemas de NLP.
- **Regularización avanzada:** `recurrent_dropout` (mismo dropout en todos los timesteps), `weight decay`.
- **Feature engineering:** ¿podemos agregar variables como el mes del año, la hora del día?

---

### Ejercicios propuestos

1. **Cambiá `FAST_MODE = False`** y entrenás más épocas. ¿Cuánto mejoran los modelos?
2. **Aumentá `HIDDEN` a 64 o 128** en la LSTM. ¿Cuándo empieza a overfittear?
3. **Implementá un `ManualLSTMModel` bidireccional** procesando la secuencia en orden inverso y promediando con el modelo normal.
4. **Probá `sequence_length = 240`** (10 días de historia). ¿Mejora el pronóstico?
5. **Modificá `ManualLSTMModel`** para devolver los estados en cada timestep y visualizarlos (análogo a la visualización de la RNN en `rnn_example.ipynb`).